In [0]:
import pandas as pd
import glob
import os
from pyspark.sql.functions import col, lit, stack, to_timestamp, date_format, hour, minute, when, lower, initcap
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

### Household Treatment

In [0]:
%sql
CREATE OR REPLACE TABLE smart_meters.silver.dim_household AS
    SELECT * 
    FROM 
        smart_meters.bronze.households
    WHERE Acorn_grouped IN 
        ('Affluent', 'Adversity', 'Comfortable');

### Holidays Treatment

In [0]:
%sql
CREATE OR REPLACE TABLE smart_meters.silver.dim_holidays_silver AS
WITH base_holidays AS (
  -- 1. Get dates from Bronze and cast to date type
  SELECT 
    CAST(Bank_holidays AS DATE) AS date, 
    'Holiday' AS type
  FROM smart_meters.bronze.holidays
),
all_dates AS (
  -- 2. Combine Holidays with calculated Eves (date - 1)
  SELECT date, type FROM base_holidays
  UNION ALL
  SELECT date_sub(date, 1) AS date, 'Holiday Eve' AS type FROM base_holidays
),
ranked_dates AS (
  -- 3. Resolve conflicts: If a date has two types, 'Holiday' "wins"
  SELECT 
    date, 
    type,
    ROW_NUMBER() OVER (PARTITION BY date ORDER BY type ASC) as rank
  FROM all_dates
)
-- 4. Final selection of unique dates
SELECT date, type
FROM ranked_dates
WHERE rank = 1
ORDER BY date;

select * from smart_meters.silver.dim_holidays_silver limit 10

### Tariffs Treatment

In [0]:
%sql
CREATE TABLE IF NOT EXISTS smart_meters.silver.dim_tariffs AS
SELECT 
    CAST(tariff_id AS INT) AS tariff_id,
    CAST(type AS STRING) AS type,
    CAST(period AS STRING) AS period,
    CAST(price AS DOUBLE) AS price
FROM smart_meters.bronze.dim_tariffs
WHERE price IS NOT NULL;

SELECT * FROM smart_meters.silver.dim_tariffs LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE smart_meters.silver.dim_tariffs_silver AS
SELECT
    *
FROM smart_meters.bronze.tariffs;
-- WHERE minute(to_timestamp(TariffDateTime, "d/M/yyyy H:mm")) = 0
-- ORDER BY day, hour;

-- Verificar o resultado
SELECT * FROM smart_meters.silver.dim_tariffs_silver LIMIT 10;

### Silver Weather Table

In [0]:
%sql

CREATE OR REPLACE TABLE smart_meters.silver.weather_enriched AS
WITH raw_processing AS (
    SELECT 
        time AS original_time,
        summary,        
        CASE 
        -- We only check for the first condition because case stops at first check
            WHEN apparentTemperature <= -38.0 THEN 'Unknown'
            WHEN apparentTemperature <= -25.0 THEN 'Extreme Cold Stress'
            WHEN apparentTemperature <= -13.0 THEN 'Strong Cold Stress'
            WHEN apparentTemperature < 0.0  THEN 'Moderate Cold Stress' 
            WHEN apparentTemperature < 9.0  THEN 'Slight Cold Stress'
            WHEN apparentTemperature < 26.0 THEN 'Neutral'
            WHEN apparentTemperature < 32.0 THEN 'Moderate Heat Stress'
            WHEN apparentTemperature < 38.0 THEN 'Strong Heat Stress'
            WHEN apparentTemperature >= 38.0 THEN 'Extreme Heat Stress'
            ELSE 'Unknown'
        END AS thermal_stress_level,

        CASE 
            WHEN temperature < 0.0 THEN 'Bin 01: < 0°C'
            WHEN apparentTemperature < 4.0 THEN 'Bin 02: 0°C to 4°C'
            WHEN apparentTemperature < 8.0 THEN 'Bin 03: 4°C to 8°C'
            WHEN apparentTemperature < 12.0 THEN 'Bin 04: 8°C to 12°C'
            WHEN apparentTemperature < 16.0 THEN 'Bin 05: 12°C to 16°C'
            WHEN apparentTemperature < 20.0 THEN 'Bin 06: 16°C to 20°C'
            WHEN apparentTemperature < 24.0 THEN 'Bin 07: 20°C to 24°C'
            WHEN apparentTemperature < 28.0 THEN 'Bin 08: 24°C to 28°C'
            WHEN apparentTemperature < 32.0 THEN 'Bin 09: 28°C to 32°C'
            ELSE 'Bin 10: >= 32°C'
        END AS temp_bin_10
    FROM smart_meters.bronze.weather
    WHERE summary IS NOT NULL
)
SELECT 
    original_time,
    summary,
    thermal_stress_level,
    temp_bin_10,
    -- Generate the ID once so dim_time and dim_weather stay in sync
    MD5(CONCAT_WS('|', summary, thermal_stress_level, temp_bin_10)) AS sk_weather
FROM raw_processing;

SELECT * FROM smart_meters.silver.weather_enriched LIMIT 10;

### Consumption Blocks Treatment

In [0]:
somas_sql = []
colunas_unpivot = []

for h in range(24):
    h_str = f"{h:02d}"
    col_a = f"hh_{2*h}"
    col_b = f"hh_{2*h+1}"
    
    somas_sql.append(f"{col_a} + {col_b} AS h_{h_str}")
    colunas_unpivot.append(f"h_{h_str}")

somas_string = ", ".join(somas_sql)
unpivot_string = ", ".join(colunas_unpivot)

query_final = f"""
CREATE OR REPLACE TEMP VIEW temp_bronze_pivoted AS
WITH unique_tariff_days AS (
    SELECT DISTINCT TO_DATE(to_timestamp(TariffDateTime, 'd/M/yyyy H:mm')) AS tariff_day
    FROM smart_meters.bronze.tariffs
),
base_filtered AS (
    SELECT c.* 
    FROM smart_meters.bronze.consumption c
    INNER JOIN unique_tariff_days t 
        ON c.day = t.tariff_day
),
summed_hours AS (
    SELECT 
        LCLid, 
        day,
        {somas_string}
    FROM base_filtered
)
SELECT 
    LCLid, 
    day, 
    REPLACE(hora, 'h_', '') AS hour, 
    consumo_kwh
FROM summed_hours
UNPIVOT (
    consumo_kwh FOR hora IN ({unpivot_string})
)
"""

spark.sql(query_final)

display(spark.sql("SELECT * FROM temp_bronze_pivoted LIMIT 10"))

#### Consumption Blocks + ID Tariffs

In [0]:
%sql
CREATE OR REPLACE TABLE smart_meters.silver.fact_consumption AS
WITH base_consumption AS (
    SELECT 
        LCLid,
        day,
        hour,
        consumo_kwh
    FROM temp_bronze_pivoted
),
enriched_with_household AS (
    SELECT 
        c.*,
        h.stdorToU,
        h.Acorn_grouped
    FROM base_consumption c
    LEFT JOIN smart_meters.silver.dim_household h 
        ON c.LCLid = h.LCLid
),
mapped_tariffs AS (
    SELECT 
        e.*,
        CASE 
            WHEN e.stdorToU = 'Std' THEN 1
            WHEN e.stdorToU = 'ToU' AND lower(t.Tariff) = 'low'    THEN 2
            WHEN e.stdorToU = 'ToU' AND lower(t.Tariff) = 'normal' THEN 3
            WHEN e.stdorToU = 'ToU' AND lower(t.Tariff) = 'high'   THEN 4
            ELSE NULL 
        END AS tariff_id
    FROM enriched_with_household e
    LEFT JOIN smart_meters.silver.dim_tariffs_silver t 
        ON TO_DATE(to_timestamp(t.TariffDateTime, 'd/M/yyyy H:mm')) = e.day 
        AND CAST(date_format(to_timestamp(t.TariffDateTime, 'd/M/yyyy H:mm'), 'H') AS STRING) = e.hour
)
SELECT 
    m.consumo_kwh AS energy_spent,
    m.tariff_id,
    m.LCLid AS dim_household_lclid,
    CAST(CONCAT(regexp_replace(m.day, '-', ''), lpad(m.hour, 2, '0')) AS BIGINT) AS time_id,
    m.day,
    m.hour
FROM mapped_tariffs m;

SELECT * FROM smart_meters.silver.fact_consumption LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE smart_meters.silver.fact_consumption AS
WITH base_consumption AS (
    SELECT 
        LCLid,
        day,
        hour,
        consumo_kwh
    FROM temp_bronze_pivoted
),
enriched_with_household AS (
    SELECT 
        c.*, -- Select all columns from base consumption
        h.stdorToU,
        h.Acorn_grouped
    FROM base_consumption c
    LEFT JOIN smart_meters.silver.dim_household h 
        ON c.LCLid = h.LCLid
),
with_tariff_period AS (
    SELECT 
        e.*,
        t.Tariff AS tariff_period
    FROM enriched_with_household e
    LEFT JOIN smart_meters.bronze.tariffs t 
        ON e.day = TO_DATE(to_timestamp(t.TariffDateTime, 'd/M/yyyy H:mm'))
        AND e.hour = CAST(date_format(to_timestamp(t.TariffDateTime, 'd/M/yyyy H:mm'), 'H') AS STRING)
),
mapped_tariffs AS (
    SELECT 
        tp.*,
        dt.tariff_id
    FROM with_tariff_period tp
    LEFT JOIN smart_meters.silver.dim_tariffs dt
        ON tp.stdorToU = dt.type
        AND (tp.stdorToU = 'Std' OR lower(tp.tariff_period) = lower(dt.period))
)

SELECT DISTINCT
    m.consumo_kwh AS energy_spent,
    m.tariff_id,
    m.LCLid AS dim_household_lclid,
    CAST(CONCAT(regexp_replace(m.day, '-', ''), lpad(m.hour, 2, '0')) AS BIGINT) AS time_id,
    m.day,
    m.hour
FROM mapped_tariffs m;

SELECT * FROM smart_meters.silver.fact_consumption LIMIT 10;

### Dim_Time Creation (From Weather)

In [0]:
%sql
CREATE OR REPLACE TABLE smart_meters.silver.dim_time AS
    SELECT 
        CAST(date_format(w.time, 'yyyyMMddHH') AS BIGINT) AS time_id,
        
        year(w.time) AS year,
        month(w.time) AS month,
        day(w.time) AS day,
        hour(w.time) AS hour,
        w.time AS date_time,
        we.sk_weather,
        
        CASE 
            WHEN month(w.time) IN (3, 4, 5) THEN 'Spring'
            WHEN month(w.time) IN (6, 7, 8) THEN 'Summer'
            WHEN month(w.time) IN (9, 10, 11) THEN 'Autumn'
            ELSE 'Winter' 
        END AS season,
        
        COALESCE(h.type, 'None') AS holiday_type
        
    FROM 
        smart_meters.bronze.weather w
    
    LEFT JOIN smart_meters.silver.weather_enriched we
        ON w.time = we.original_time

    LEFT JOIN smart_meters.silver.dim_holidays_silver h 
        ON CAST(w.time AS DATE) = h.date;

SELECT * 
FROM smart_meters.silver.dim_time
WHERE holiday_type != "None"
ORDER BY time_id
LIMIT 10